<img src="logo.png" alt="Vegeta" width="240">

# Kestrel — the test campaign: every CFD case, failures accepted

Notebook 38 designs Kestrel-4Q in a loop and plugs the CFD in where it exists; **this notebook is the CFD.** It
prepares and runs every Aeromant case the study asks for — Kestrel-4Q (Part I's airframe) and Kestrel-4Q-2 (elevons,
faired posts, no diagonals), each at its transition points with the four propellers as rotor disks, in the hover, and
bare from 0 to 90°; and the two finalist sections as extruded slabs — at the quality it is given, and writes what
solved to `designs/data/kestrel_cfd_results.json`, which notebook 38 reads on its next run (iteration 3, § 7).

**A failure is a result.** A case that fails in a step, or runs its iterations into nonsense, is listed as `FAILED` or
`diverged` with its reason, counted, and left out of the correction; nothing here raises on it. The first campaign
(screening, 400 iterations) solved the ten bare cases of Kestrel-4Q and lost all fourteen blown ones to the solver's
numerics — those numbers are in the results file now and this notebook is how they get replaced.

How to run it:

| variable | default | meaning |
|---|---|---|
| `VEGETA_SKIP_OPENFOAM=1` | unset | list and prepare only: every case NOT RUN (what this copy shows) |
| `KESTREL_CFD_QUALITY` | `screening` | `screening` (~0.3–0.5 M cells, 400 iterations) or `fine` (×3 cells, 800 iterations) |
| `KESTREL_CFD_JOBS` / `KESTREL_CFD_PROCESSORS` | 1 / 4 | cases in parallel / MPI ranks per case |
| `KESTREL_CFD_ONLY` | unset | `4Q-1`, `4Q-2`, `sections`, or a comma list: run those campaigns only |
| `KESTREL_CFD_SWEEP_ONLY=1` | unset | the bare sweeps only (no rotor disks): the cheap half |

The cases are cached next to the notebook (`40_kestrel_tests.cache/`): a second run solves only what is new.
**Run notebook 41 (`prop_lab`) first**: its calibration of the 9×6 — the blade section the rotor disks are fed, the
drive tables, the propeller normal force — is read here and in notebook 38 when it is on file.

In [ ]:
import json, math, os, shutil, sys, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import aeromant
sys.path.insert(0, "designs")
import kestrel, kestrel_systems as ks, kestrel_flight as kf, kestrel_transition as kt, kestrel_cfd as kcfd, kestrel_drawings as kd, kestrel_loop as kl

pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 220)
RUNS = Path("_runs/kestrel_tests"); RUNS.mkdir(parents=True, exist_ok=True)
OUT = Path("output/40_kestrel_tests"); OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUALITY = os.environ.get("KESTREL_CFD_QUALITY", "screening")
JOBS, PROCS = int(os.environ.get("KESTREL_CFD_JOBS", "1")), int(os.environ.get("KESTREL_CFD_PROCESSORS", "4"))
ONLY = [s.strip() for s in os.environ.get("KESTREL_CFD_ONLY", "4Q-1,4Q-2,sections").split(",") if s.strip()]
SWEEP_ONLY = os.environ.get("KESTREL_CFD_SWEEP_ONLY") == "1"
TODAY = datetime.date.today().isoformat()
print(f"CFD {'on' if RUN_CFD else 'NOT RUN (VEGETA_SKIP_OPENFOAM=1)'} | quality {QUALITY} ({kcfd.QUALITY[QUALITY]}) | jobs {JOBS} × {PROCS} ranks | campaigns {ONLY}"
      + (" | bare sweeps only" if SWEEP_ONLY else ""))
PL = ks.prop_lab_calibration()
print("prop_lab (notebook 41):", (f"calibration on file — cl x{PL['cl_scale']:.3f}, cd x{PL['cd_scale']:.3f}, CNα x{PL['cn_factor']:.3f}: the rotor disks' polar and the drive tables carry it"
                                  if PL["on_file"] else "NOT RUN: the rotor disks run on Boreas's generic blade section"))
try:
    import kestrel_section as ksec
    HAVE_ZEPHYR = True
except Exception as exc:
    HAVE_ZEPHYR = False; print("Zephyr not available:", type(exc).__name__)

## 1. The aircraft and their transition points

The same vehicles as notebook 38: Kestrel-4Q from the default parameters, Kestrel-4Q-2 from the Part II options (the
CAD's wings are NACA 0012 in both — the reflexed section is the slabs' question, not the airframe's). The CFD points
come from each aircraft's nominal transitions: the first time the free-stream angle of attack passes 80, 60, 45, 30,
20, 15 and 10° going forward and coming back — 4Q-2's back transition entered at 13 m/s, its operating rule — plus
the **hover**, the level **cruise** at 16 m/s and the **stall** at 1.05 × the stall speed (the five states the
campaign is asked for: cruise, stall, hover, vertical → horizontal, horizontal → vertical), and the bare sweep at 10 m/s — thirteen angles, with 8, 12, 14 and 16° added to the design notebook's ten
so that the stall's peak is on the grid (the first campaign's 5° steps could not tell the model's CL max of 0.81 at
12° from the CFD's 0.54 at 15°).

In [ ]:
P = kestrel.resolve(); A = kf.aero(P)
VEH = kt.make_vehicle(P, aero=A, name="Kestrel-4Q")
P2 = kestrel.resolve(dict(P, elevons=True, strut_fairing=True, bracing="posts"))
D2 = kf.drag_buildup(P2, faired=True)
VEH2 = kt.make_vehicle(P2, aero=A, pol=kf.polar(P2, a=A, d=D2), name="Kestrel-4Q-2")
NOM1 = [kt.simulate(VEH, "forward", t_tr=3.0), kt.simulate(VEH, "back", t_tr=3.0)]
NOM2 = [kt.simulate(VEH2, "forward", t_tr=3.0), kt.simulate(VEH2, "back", t_tr=3.0, V_entry=13.0)]
PTS = {"4Q-1": kcfd.points(NOM1, mass_kg=VEH.mass) + kcfd.flight_points(VEH), "4Q-2": kcfd.points(NOM2, mass_kg=VEH2.mass) + kcfd.flight_points(VEH2)}
VEHS = {"4Q-1": (VEH, P), "4Q-2": (VEH2, P2)}
for k, (v, p_) in VEHS.items():
    print(f"{k}: {v.mass:.3f} kg, CG {v.x_cg * 1000:.0f} mm, Cd0 {v.pol.cd0:.4f}, {len(PTS[k])} transition points")
display(pd.DataFrame({k: ep.metrics for k, ep in zip(("4Q-1 forward", "4Q-1 back", "4Q-2 forward", "4Q-2 back (13 m/s)"), NOM1 + NOM2)}).T[["height loss [m]", "height gain [m]", "pitch error max [deg]", "pass"]])

## 2. The case list

Everything the campaign holds, with its template, air speed and rotor speeds, before a single solver starts. The
sections' slabs are the two finalists of notebook 38 § 14 (NACA 0012 and the reflexed NACA 23112 with k₂/k₁ 0.03)
at four angles, AR 8, 16 m/s.

In [ ]:
SPECS = {k: kcfd.specs(PTS[k], sweep=True, sweep_alphas=kcfd.SWEEP_ALPHAS_FINE) for k in VEHS}      # the stall's peak resolved
if SWEEP_ONLY:
    SPECS = {k: {n: s for n, s in v.items() if not s["props"]} for k, v in SPECS.items()}
LISTS = {k: kcfd.not_run_table(v) for k, v in SPECS.items()}
for k, t in LISTS.items():
    print(f"{k}: {len(t)} cases ({int(t['template'].eq('aircraft_quad_rotor_disks').sum())} with rotor disks)"); display(t)
FINALISTS = [("NACA 0012", None), ("NACA 23112", 0.03)]
SSPEC = ksec.cfd_specs(FINALISTS) if HAVE_ZEPHYR else {}
SLIST = kcfd.section_not_run_table(SSPEC) if SSPEC else pd.DataFrame()
print(f"sections: {len(SLIST)} slab cases"); display(SLIST)
n_cases = sum(len(t) for t in LISTS.values()) + len(SLIST)
cells = {"screening": 0.4e6, "fine": 1.2e6}[QUALITY]
print(f"{n_cases} cases at {QUALITY}: ~{n_cases * cells / 1e6:.0f} M cells in all, {kcfd.QUALITY[QUALITY]['iterations']} iterations each — "
      f"order of {n_cases * (0.25 if QUALITY == 'screening' else 1.0) / JOBS:.0f} h on {PROCS} ranks per case")

## 3. The campaigns

One function runs a campaign: plan the cases (the STL of the airframe at each angle, the four disks at Boreas's rpm),
run them through the cache, build the table — `solved`, `diverged`, `FAILED` or `NOT RUN` per case — print the
summary and write the solved cases into the results file under the campaign's key. A campaign that solved nothing
leaves the previous record for that key alone.

In [ ]:
def campaign(key, veh, p_, spec):
    T = kcfd.not_run_table(spec)
    if RUN_CFD and key in ONLY:
        try:
            cases = kcfd.plan(RUNS / key, PTS[key], x_cg_m=veh.x_cg, z_cg_m=veh.z_cg, quality=QUALITY, sweep=True, p=p_, sweep_alphas=kcfd.SWEEP_ALPHAS_FINE)
            cases = {n: c for n, c in cases.items() if n in spec}
            names = list(cases)
            res = aeromant.run_cases([cases[n] for n in names], jobs=JOBS, processors=PROCS, progress=True, cache=True)
            T = kcfd.table(dict(zip(names, res)), spec, p_)
        except Exception as exc:                        # no OpenFOAM, no CAD kernel, a broken environment: the list stays NOT RUN
            print(f"{key}: CFD not run — {type(exc).__name__}: {str(exc)[:200]}")
    sm = kcfd.summary(T)
    print(f"{key}: {sm['line']}")
    kcfd.save_records(T, kcfd.RESULTS_JSON, key=key, meta={"source": "40_kestrel_tests", "quality": QUALITY, "aircraft": veh.name, "date": TODAY,
                                                           "iterations": kcfd.QUALITY[QUALITY]["iterations"]})
    T.to_csv(OUT / f"cfd_{key}.csv")
    return T
TABLES = {}
TABLES["4Q-1"] = campaign("4Q-1", VEH, P, SPECS["4Q-1"]); display(TABLES["4Q-1"])

In [ ]:
TABLES["4Q-2"] = campaign("4Q-2", VEH2, P2, SPECS["4Q-2"]); display(TABLES["4Q-2"])

In [ ]:
STAB = SLIST
if SSPEC and RUN_CFD and "sections" in ONLY:
    try:
        scases = {k: kcfd.section_case(RUNS / "sections", v, quality=QUALITY) for k, v in SSPEC.items()}
        names = list(scases)
        res = aeromant.run_cases([scases[n] for n in names], jobs=JOBS, processors=PROCS, progress=True, cache=True)
        STAB = kcfd.section_table(dict(zip(names, res)), SSPEC)
    except Exception as exc:
        print(f"sections: CFD not run — {type(exc).__name__}: {str(exc)[:200]}")
if len(STAB):
    st = STAB["status"].astype(str)
    print("sections:", ", ".join(f"{int((st == k).sum())} {k}" for k in ("solved", "diverged", "FAILED", "NOT RUN")))
    kcfd.save_section_table(STAB, kcfd.RESULTS_JSON, key="sections", meta={"source": "40_kestrel_tests", "quality": QUALITY, "date": TODAY})
    STAB.to_csv(OUT / "cfd_sections.csv")
    display(STAB)
    if HAVE_ZEPHYR and "cl section estimate" in STAB.columns:                       # the slab against NeuralFoil
        ST_ = ksec.table()
        rows = {}
        for k, r in STAB[STAB["status"] == "solved"].iterrows():
            sec = [c for c in FINALISTS if r["section"].startswith(c[0]) and ((c[1] is None) == ("k₂" not in r["section"]))]
            if sec:
                pc = ST_[ksec.key(*sec[0])]["polar_cruise"]
                rows[k] = {"α [deg]": r["α [deg]"], "cl slab → section": r["cl section estimate"], "cl NeuralFoil": float(np.interp(r["α [deg]"], pc["alpha_deg"], pc["cl"])),
                           "cd slab (3-D)": r["CD"], "cd NeuralFoil (2-D)": float(np.interp(r["α [deg]"], pc["alpha_deg"], pc["cd"])), "Cm slab": r["Cm (c/4, chord)"],
                           "cm NeuralFoil": float(np.interp(r["α [deg]"], pc["alpha_deg"], pc["cm"]))}
        if rows:
            display(pd.DataFrame(rows).T.round(4))

## 4. Plug in and fly again

What notebook 38 does with the results, done here first so the campaign reads its own answer: for each aircraft the
bare sweep against the model's polar (lift slope, CL max, Cd0, the static margin from Cmα), the correction — from the
blown cases if any solved, in α and the slipstream ratio μ; else from the sweep, in α — and the transition matrix flown
on it beside the model's.

In [ ]:
VERDICTS = {}
for key, (veh, p_) in VEHS.items():
    print(f"\n===== {key}")
    PI = kcfd.plug_in(veh, kcfd.RESULTS_JSON, key)
    camp = PI["campaign"]
    if not camp:
        print("no campaign on file"); continue
    print("campaign:", camp.get("meta")); display(pd.Series(camp["summary"], name="cases").to_frame().T)
    RT = kcfd.records_table(camp["records"]) if camp["records"] else None
    if RT is not None:
        cmp = kcfd.sweep_comparison(RT, veh)
        if len(cmp):
            display(cmp.drop(columns=["note"]).astype({"α [deg]": float}).round(3)); display(pd.DataFrame(kcfd.sweep_numbers(cmp)).round(3))
            kd.polar_figure(veh.pol, RT); plt.show()
    if PI["airframe"] is None:
        print("nothing usable solved: the transitions stand on the model"); continue
    corr = PI["airframe"]
    print(f"correction from the {corr.source}, {len(corr.samples)} points")
    vc = kt.make_vehicle(p_, aero=A, pol=veh.pol, airframe=corr, name=f"{veh.name}, CFD aero")
    kw = {"V_entry": 13.0} if key == "4Q-2" else {}
    mats = {}
    for lab, v_ in (("model", veh), (f"CFD ({corr.source})", vc)):
        m_, _ = kt.matrix({lab: v_}, durations=(3.0, 5.0), headwinds=(0.0, 6.0), kinds=("forward",), progress=False)
        mb, _ = kt.matrix({lab: v_}, durations=(3.0, 5.0), headwinds=(0.0, 6.0), kinds=("back",), progress=False, **kw)
        mats[lab] = pd.concat([m_, mb])
    MATK = pd.concat(mats.values())
    cols = ["height loss [m]", "height gain [m]", "pitch error max [deg]", "time stalled, free stream [s]", "moment shortfall [s]", "roll demand / authority", "pass"]
    display(MATK[cols].astype({c: float for c in cols if c != "pass"}).round(2))
    VERDICTS[key] = kt.verdict(MATK)
    for k, v in VERDICTS[key].items():
        print(f"{k}: {v['passes']} of {v['of']} pass" + ("".join(f"\n   - {w}" for w in v["failures"]) if v["failures"] else ""))
    MATK.to_csv(OUT / f"transitions_{key}.csv")

## 5. The campaign's record

What is on file for notebook 38 to read, campaign by campaign, and every case that did not solve with its reason —
the list to work through before the next run.

In [ ]:
DOC = kcfd.load_records(kcfd.RESULTS_JSON) or {"campaigns": {}}
REC = pd.DataFrame({k: dict(v.get("summary", {}), **{"source": v.get("meta", {}).get("source"), "quality": v.get("meta", {}).get("quality"), "date": v.get("meta", {}).get("date")})
                    for k, v in DOC["campaigns"].items()}).T
display(REC)
FAILS = {f"{k}: {n}": s for k, v in DOC["campaigns"].items() for n, s in v.get("statuses", {}).items() if s["status"] in ("FAILED", "diverged")}
if FAILS:
    print(f"{len(FAILS)} cases did not solve:"); display(pd.DataFrame(FAILS).T)
else:
    print("every case that ran solved" if any(v.get("summary", {}).get("solved") for v in DOC["campaigns"].values()) else "nothing has run yet")
CHK = [kl.check("campaigns on file", len(DOC["campaigns"]), "≥", 3),
       kl.check("4Q-1: blown cases solved", DOC["campaigns"].get("4Q-1", {}).get("summary", {}).get("solved with rotor disks", 0), "≥", 1),
       kl.check("4Q-1: bare sweep solved", DOC["campaigns"].get("4Q-1", {}).get("summary", {}).get("solved bare", 0), "≥", 10),
       kl.check("4Q-2: blown cases solved", DOC["campaigns"].get("4Q-2", {}).get("summary", {}).get("solved with rotor disks", 0), "≥", 1),
       kl.check("4Q-2: bare sweep solved", DOC["campaigns"].get("4Q-2", {}).get("summary", {}).get("solved bare", 0), "≥", 10),
       kl.check("sections: slabs solved", DOC["campaigns"].get("sections", {}).get("summary", {}).get("solved", 0), "≥", 8)]
display(kl.check_table(CHK, "the campaign"))
print("the file:", kcfd.RESULTS_JSON, "— notebook 38 reads it on its next run (§ 7, iteration 3).")

## 6. Findings

*(written after the first campaign with the new numerics)* What is on file today is the first campaign's bare sweep of
Kestrel-4Q (ten cases, screening, 400 iterations, none at the residual target): the model's lift slope within 2 %, its
zero-lift drag a fifth conservative, its stall harsher than the CFD's (CL 0.54–0.57 held from 15° to 30°). Nothing
has solved with rotor disks yet — the question the campaign exists for — and nothing for Kestrel-4Q-2 or the sections.
Run this notebook where OpenFOAM is installed; begin with `KESTREL_CFD_SWEEP_ONLY=1` (the cheap half, an hour) and
then the blown cases at screening quality; a blown case that still fails is the solver's log to read
(`_runs/kestrel_tests/<campaign>/<case>/log.*`), not a number to doubt.